## Probabilistic modeling

This notebook shows how to add uncertainty to a structural geological model.

Input data are never exact: the position of a fault or its dip angle are only known within some range. Instead of a single model, we describe uncertain parameters with **probability distributions**, draw many random samples from them and compute one model per sample (a **Monte Carlo simulation**). The resulting set of models (*realizations*) shows where the geology is well constrained and where it is uncertain.

**Workflow in this notebook:**
1. Build a simple model with a fault
2. Define a stochastic model: fault position and dip as random variables
3. Compute many realizations
4. Save and load the results
5. Visualize the uncertainty: realizations, lithology probabilities and entropy

In addition to GemPy, it requires [PyTorch](https://pytorch.org/) and [Pyro](https://pyro.ai/). Pyro is a probabilistic programming library. Here we only use it to draw random samples, but it is also the basis for Bayesian inference with GemPy (see the outlook at the end).

In [ ]:
# Importing GemPy
import gempy as gp
import gempy_viewer as gpv
from gempy_engine.config import AvailableBackends

# Auxiliary libraries
import contextlib
import io
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Probabilistic modeling
import pyro
import pyro.distributions as dist
import torch

### 1) Original model

A simple 2D model (one cell thick in y-direction) with two layers offset by a fault. We compute it with the PyTorch backend, which GemPy needs for probabilistic inference with Pyro.

In [ ]:
# Resolution: we need it later to reshape the results
resolution = [50, 3, 50]

# Create a GeoModel instance
geo_model = gp.create_geomodel(
    project_name='Prob_Model',
    extent=[0, 1000, 490, 510, 0, 1000],
    resolution=resolution,
    importer_helper=gp.data.ImporterHelper(
        path_to_orientations="data/prob_example_orientations.csv",
        path_to_surface_points="data/prob_example_surface_points.csv"
    )
)

# Map geological series to surfaces
gp.map_stack_to_surfaces(
    gempy_model=geo_model,
    mapping_object={
        "Fault_Series": ('fault'),
        "Strat_Series": ('rock2', 'rock1')
    }
)

# Define youngest structural group as fault
gp.set_is_fault(geo_model, ["Fault_Series"])

# Use the PyTorch backend
engine_config = gp.data.GemPyEngineConfig(backend=AvailableBackends.PYTORCH)

# Compute the model
gp.compute_model(geo_model, engine_config=engine_config)

In [ ]:
# 2D visualization of the model
gpv.plot_2d(geo_model, show_boundaries=False, show_lith=True)

The fault is defined by a single surface point and a single orientation, both the first entries (index 0) in the input data: located at x = 500 m, dipping 60°.

In [ ]:
# Input data: surface points and orientations (index 0 = fault)
display(geo_model.surface_points_copy.df.head())
display(geo_model.orientations_copy.df.head())

### 2) Stochastic model

We treat two fault parameters as uncertain and describe each with a normal distribution:

| Parameter | Mean | Standard deviation |
|---|---|---|
| Fault position (x) | 500 m | 50 m |
| Fault dip | 60° | 10° |

The function below draws one sample of both parameters with `pyro.sample`, updates the fault's surface point and orientation in the model, and recomputes it.

In [ ]:
# Reproducible random samples
pyro.set_rng_seed(42)

def stochastic_geomodel():
    """Draw one sample of the fault parameters, update the model and recompute it."""
    # Draw random values for the uncertain parameters
    fault_x = pyro.sample("fault_x", dist.Normal(500.0, 50.0)).item()
    fault_dip = pyro.sample("fault_dip", dist.Normal(60.0, 10.0)).item()

    # Move the fault's surface point
    gp.modify_surface_points(geo_model, slice=0, X=fault_x)

    # Move the fault's orientation and change its dip.
    # GemPy stores orientations as pole vectors (G_x, G_y, G_z). For a fault striking
    # in y-direction (azimuth 90°), the pole vector is (sin(dip), 0, cos(dip)).
    dip_rad = np.radians(fault_dip)
    gp.modify_orientations(geo_model, slice=0, X=fault_x,
                           G_x=np.sin(dip_rad), G_y=0.0, G_z=np.cos(dip_rad))

    # Recompute the model (GemPy's progress messages are hidden)
    with contextlib.redirect_stdout(io.StringIO()):
        gp.compute_model(geo_model, engine_config=engine_config)

    return fault_x, fault_dip

In [ ]:
# Generate one realization
fault_x, fault_dip = stochastic_geomodel()
print(f"Sampled fault position: {fault_x:.1f} m")
print(f"Sampled fault dip:      {fault_dip:.1f}°")

gpv.plot_2d(geo_model)

### 3) Multiple realizations

Now we repeat this many times and store, for every realization, the sampled parameters, the lithology and the scalar fields in our 2D section (the middle cell in y-direction). The scalar fields and their values at the surface points let us draw the fault and layer boundaries later.

To speed things up, we switch off the extraction of 3D surface meshes, which we don't need here. This makes every realization about 5 times faster. 100 realizations take less than a minute.

In [ ]:
# Surface meshes are not needed for the realizations: switch off for speed
geo_model.interpolation_options.mesh_extraction = False

def section(array):
    """Extract the middle y-slice of a full-grid array as a 2D (z, x) array."""
    return array.reshape(resolution)[:, 1, :].T

In [ ]:
n_realizations = 100

fault_x_values = np.empty(n_realizations)
fault_dip_values = np.empty(n_realizations)
lith_sections = np.empty((n_realizations, resolution[2], resolution[0]))
fault_field_sections = np.empty((n_realizations, resolution[2], resolution[0]))
strat_field_sections = np.empty((n_realizations, resolution[2], resolution[0]))
fault_levels = np.empty((n_realizations, 1))
strat_levels = np.empty((n_realizations, 2))

for i in tqdm(range(n_realizations)):
    fault_x_values[i], fault_dip_values[i] = stochastic_geomodel()

    raw = geo_model.solutions.raw_arrays
    lith_sections[i] = section(raw.lith_block)
    # Scalar fields: index 0 = fault, index 1 = stratigraphic series
    fault_field_sections[i] = section(raw.scalar_field_matrix[0])
    strat_field_sections[i] = section(raw.scalar_field_matrix[1])
    # Scalar field values at the surfaces (used as contour levels)
    fault_levels[i] = raw.scalar_field_at_surface_points[0]
    strat_levels[i] = raw.scalar_field_at_surface_points[1]

### 4) Saving and loading the results

Saving the realizations means you can analyze them later without recomputing. The file is written next to the input data (and not tracked by git).

In [ ]:
filename = "data/my_uq_results.npz"

# Save all realizations
np.savez_compressed(
    filename,
    fault_x_values=fault_x_values,
    fault_dip_values=fault_dip_values,
    lith_sections=lith_sections,
    fault_field_sections=fault_field_sections,
    strat_field_sections=strat_field_sections,
    fault_levels=fault_levels,
    strat_levels=strat_levels,
)
print(f"Results saved to {filename}")

In [ ]:
# Load them again (e.g. in a new session)
results = np.load(filename)
print(list(results.keys()))

lith_sections = results["lith_sections"]

### 5) Visualization

**Sampled parameters.** First a check that the samples follow the distributions we defined:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
axes[0].hist(fault_x_values, bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Fault position x (m)')
axes[1].hist(fault_dip_values, bins=20, color='indianred', edgecolor='white')
axes[1].set_xlabel('Fault dip (°)')
for ax in axes:
    ax.set_ylabel('Count')
plt.show()

**Individual realizations.** Each panel shows one realization with its fault (white) and layer boundaries (black):

In [ ]:
extent = [0, 1000, 0, 1000]

def draw_boundaries(ax, i, fault_color='white', strat_color='black', linewidth=1.5):
    """Draw fault and layer boundaries of realization i as contour lines."""
    ax.contour(fault_field_sections[i], levels=fault_levels[i], extent=extent,
               colors=fault_color, linewidths=linewidth, linestyles='solid')
    ax.contour(strat_field_sections[i], levels=np.sort(strat_levels[i]), extent=extent,
               colors=strat_color, linewidths=linewidth, linestyles='solid')

fig, axes = plt.subplots(3, 3, figsize=(9, 9), constrained_layout=True, sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    ax.imshow(lith_sections[i], origin='lower', extent=extent, cmap='viridis')
    draw_boundaries(ax, i)
    ax.set_title(f"x = {fault_x_values[i]:.0f} m, dip = {fault_dip_values[i]:.0f}°", fontsize=9)
plt.show()

**Not every realization is geologically reasonable.** In some realizations the layers are not offset by the fault but bend smoothly across it. This happens when the sampled fault passes beyond one of the fixed surface points of the layers (e.g. a fault far to the right with a low dip). That point then ends up in the same fault block as the points on the other side, and GemPy can only connect them with a smooth surface.

So even moderate uncertainty in just two parameters can lead to very different, and partly geologically unreasonable, models. Recognizing and handling such cases, for example by including more data or using Bayesian inference (see the outlook), is a central part of probabilistic modeling.

**Lithology probabilities.** For every cell, we count in how many realizations it belongs to each lithology. Values close to 0 or 1 mean the lithology is certain, values in between mean it is uncertain.

In [ ]:
# Lithology ids present and their names (lith id = position in the structural frame + 1)
lith_ids = np.unique(lith_sections).astype(int)
element_names = geo_model.structural_frame.elements_names
lith_names = {lith_id: element_names[lith_id - 1] for lith_id in lith_ids}

# Probability of each lithology per cell
probabilities = {lith_id: np.mean(lith_sections == lith_id, axis=0) for lith_id in lith_ids}

fig, axes = plt.subplots(1, len(lith_ids), figsize=(4 * len(lith_ids), 3.5), constrained_layout=True)
for ax, lith_id in zip(axes, lith_ids):
    im = ax.imshow(probabilities[lith_id], origin='lower', extent=extent, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f"P({lith_names[lith_id]})")
fig.colorbar(im, ax=axes, shrink=0.8, label='Probability')
plt.show()

**Information entropy.** The entropy combines all lithology probabilities into one measure of uncertainty per cell: 0 where all realizations agree, higher where they disagree. The thin lines show the fault of every realization ("spaghetti plot").

In [ ]:
# Information entropy per cell: H = -sum(p * log2(p))
entropy = np.zeros_like(probabilities[lith_ids[0]])
for p in probabilities.values():
    entropy -= np.where(p > 0, p * np.log2(np.where(p > 0, p, 1)), 0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(entropy, origin='lower', extent=extent, cmap='magma')
for i in range(n_realizations):
    ax.contour(fault_field_sections[i], levels=fault_levels[i], extent=extent,
               colors='white', linewidths=0.5, alpha=0.4, linestyles='solid')
fig.colorbar(im, ax=ax, label='Entropy (bits)')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.set_title(f'Entropy and faults of {n_realizations} realizations')
plt.show()

### Outlook: Bayesian inference

Here we only propagated uncertainty forward: we assumed distributions for the input parameters and looked at the resulting spread of models. With **Bayesian inference**, we can go one step further and update these distributions with additional observations (e.g. a borehole that hits the fault, or geophysical data), so that only models consistent with the data remain likely.

Pyro provides the inference algorithms (e.g. MCMC), and the PyTorch backend lets GemPy compute the gradients they need. See [gempy_probability](https://github.com/gempy-project/gempy_probability) for examples.